In [1]:
# ============================================================================
# BLOCK 5: TOPIC MODELING - POLISH
# Input: 04_sentiment_data_pl.pkl
# Output: 05_topics_data_pl.pkl
# Runtime: ~15-20 minutes on CPU
# ============================================================================
%run 00_setup_and_config.ipynb

c:\Users\andre\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Libraries loaded successfully
✓ PyTorch device: CPU
✓ Timestamp: 2026-05-29 02:48:21
✓ Directory structure verified
✓ Base directory: c:\Users\andre\OneDrive\Desktop\2_Disertatie_RIPEMC_D.Flanja\Quantitative_Python\YT_Analysis
✓ Model configuration loaded
  - Polish sentiment: eevvgg/bert-polish-sentiment-politics
  - Romanian sentiment: readerbench/ro-sentiment
  - Embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
  - BERTopic topics: 8
  - Batch size: 8
✓ Checkpoint utility functions loaded
✓ Text cleaning functions loaded
✓ Visualization utility functions loaded
✓ Sentiment prediction function loaded
✓ SETUP AND CONFIGURATION COMPLETE

Ready for analysis. Next steps:
  1. Run Polish pipeline: 01_data_loading_pl.ipynb → 06_engagement_metrics_pl.ipynb
  2. Run Romanian pipeline: 01_data_loading_ro.ipynb → 06_engagement_metrics_ro.ipynb
  3. (Optional) Run comparative analysis: 08_comparative_analysis.ipynb

Checkpoint directory: c:\Users\andre\OneDrive\Des

In [2]:
# ============================================================================
# CELL 2: POLISH STOPWORDS FOR BERTOPIC
# ============================================================================

STOPWORDS_PL = [
    'i', 'w', 'z', 'do', 'na', 'się', 'że', 'to', 'jest', 'być', 'mieć',
    'był', 'była', 'było', 'będzie', 'może', 'już', 'nie', 'tak', 'ale',
    'jak', 'co', 'kto', 'gdzie', 'kiedy', 'dlaczego', 'bardzo', 'tylko',
    'jeszcze', 'więc', 'bo', 'gdy', 'jeśli', 'czy', 'lub', 'oraz', 'przed',
    'po', 'pod', 'nad', 'bez', 'dla', 'od', 'przy', 'za', 'przez', 'o',
    'a', 'u', 'tym', 'ten', 'ta', 'te', 'tego', 'tej', 'ich', 'nim',
    'mnie', 'ciebie', 'jemu', 'jej', 'nam', 'wam', 'oni', 'one', 'ona', 'ono',
    'the', 'and', 'of', 'in', 'to', 'a', 'is', 'it', 'for', 'on', 'with',
    'as', 'at', 'by', 'an', 'be', 'are', 'was', 'were', 'been', 'have', 'has'
]

print(f'✓ Loaded {len(STOPWORDS_PL)} Polish stopwords')


✓ Loaded 89 Polish stopwords


In [3]:
# CELL 1: Load previous checkpoint
print('='*70)
print('BLOCK 5: TOPIC MODELING - POLISH')
print('='*70)

if check_checkpoint_exists('pl', '05_topics_data'):
    df_pl = load_checkpoint('pl', '05_topics_data')
    topic_model = None
    print('✓ Loading from topics checkpoint (skipping modeling)')
else:
    df_pl = load_checkpoint('pl', '04_sentiment_data')
    if df_pl is None:
        raise FileNotFoundError('Run 03_sentiment_analysis_pl.ipynb first')

print(f'\nRunning BERTopic on {len(df_pl):,} Polish comments...')
print(f'Target topics: {BERTOPIC_CONFIG["n_topics"]}')
print(f'Embedding model: {EMBEDDING_MODEL}')

BLOCK 5: TOPIC MODELING - POLISH
✓ Loading checkpoint: 05_topics_data_pl.pkl
✓ Loading from topics checkpoint (skipping modeling)

Running BERTopic on 6,763 Polish comments...
Target topics: 8
Embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


In [4]:
# CELL 2: Load embedding model
print('\nLoading embedding model...')
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
embedding_model = embedding_model.to('cpu')
print('✓ Embedding model loaded')


Loading embedding model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2987.37it/s]


✓ Embedding model loaded


In [5]:
# ============================================================================
# CELL 3: Run BERTopic (UPDATED WITH STOPWORDS)
# ============================================================================
if not check_checkpoint_exists('pl', '05_topics_data'):
    print('\nInitializing BERTopic...')
    
    topic_model = BERTopic(
        language='multilingual',
        embedding_model=embedding_model,
        min_topic_size=15,              # ← Increased from 10
        nr_topics=8,                    # ← Target ~8 topics
        calculate_probabilities=BERTOPIC_CONFIG['calculate_probabilities'],
        verbose=True,
        n_gram_range=(2, 3),            # ← Focus on bi/trigrams
        stop_words=STOPWORDS_PL         # ← KEY: Polish stopwords
    )
    
    print('\nFitting BERTopic model...')
    texts = df_pl['clean_text'].fillna('').astype(str).tolist()
    topics, probs = topic_model.fit_transform(texts)
    
    df_pl['topic'] = topics
    df_pl['topic_probability'] = probs.max(axis=1)
    
    topic_model.save(PROCESSED_DIR / 'bertopic_model_pl')
    print(f'✓ Topic model saved')
    
    actual_topics = len(topic_model.get_topic_info())
    print(f'\n✓ Actual topic count: {actual_topics}')
    
    print('\n--- TOPIC PREVIEW ---')
    topic_info = topic_model.get_topic_info()
    for i, row in topic_info.head(8).iterrows():
        print(f"Topic {row['Topic']}: {row['Name']}")
    
else:
    topic_model = BERTopic.load(PROCESSED_DIR / 'bertopic_model_pl')
    print('✓ Topic model loaded from checkpoint')


✓ Topic model loaded from checkpoint


In [6]:
# CELL 4: Topic summary
print('\n' + '='*70)
print('TOPIC SUMMARY - POLISH')
print('='*70)

topic_info = topic_model.get_topic_info()
display(topic_info)

# Save topic summary
topic_info.to_csv(OUTPUT_DIR / 'polish' / 'pl_topics_summary.csv', index=False)
print(f"\n✓ Saved: {OUTPUT_DIR / 'polish' / 'pl_topics_summary.csv'}")


TOPIC SUMMARY - POLISH


,Topic,Count,Name,Representation,Representative_Docs
0,-1,2985,-1_nie_to_na_się,"[nie, to, na, się, że, jest, jak, do, tak, co]",[Nie ma to jak być obiektywnym i nakręcać dalej ludzi na siebie... Tu nie chodzi już o wygrywani...
1,0,1775,0_nie_to_się_na,"[nie, to, się, na, że, jest, jak, do, pan, po]",[W czasie kampanii szczególną uwagę zwróciłem na Macieja Maciaka ponieważ jedna osoba z mojej ro...
2,1,857,1_dziękuję_za_bardzo_panie,"[dziękuję, za, bardzo, panie, rafale, pozdrawiam, punkt, komentarz, pana, jak]","[Dziękuję Panie Rafale, Dziękuję Panie Rafale, za jak zwykle znakomity komentarz., Dziękuję Pani..."
3,2,833,2_to_nie_na_się,"[to, nie, na, się, że, jest, polski, do, polska, jak]","[Napisze glupie zdanie ,ale czy na pewno? > to co sie dzieje to nie jest nasza wina ,ale tez jes..."
4,3,180,3_zero_nie_kanał_się,"[zero, nie, kanał, się, to, jest, na, tv, pan, za]","[Panie Jakubie, bardzo dobry i potrzebny komentarz. Chciałbym by przedstawił go Pan na kanale Ze..."
5,4,84,4_siedem_słów_komentarz_algorytmy,"[siedem, słów, komentarz, algorytmy, algorytm, dla, siedmiu, słowo, sześć, trzy]","[Dzięki bardzo Siedem słów żeby nakarmić algorytm, Miało być siedem słów i jest siedem. Pozdrawi..."
6,5,32,5_koń_konia_koniu_to,"[koń, konia, koniu, to, jak, giertych, ciągnie, się, cecha, ohydny]","[Giertych to koń trojański PO, Koń jak to koń ciągnie! Ciągnie Wariatów z KO na dno., Koń, jak t..."
7,6,17,6_alkoholu_na_alkohol_papierosów,"[alkoholu, na, alkohol, papierosów, do, lub, się, nie, to, akcyzy]",[Jestem ZA deligitymacją alkoholu i ograniczeniem jego sprzedaży decyzjami administracyjnymi lub...



✓ Saved: outputs\polish\pl_topics_summary.csv


In [7]:
# CELL 5: Topic visualization
print('\nGenerating topic visualization...')

fig, ax = plt.subplots(figsize=(14, 8))

# Get topic sizes
topic_sizes = topic_model.get_topic_info().sort_values('Count', ascending=False)

colors = plt.cm.Set3(np.linspace(0, 1, len(topic_sizes)))
bars = ax.barh(range(len(topic_sizes)), topic_sizes['Count'].values, color=colors)

ax.set_yticks(range(len(topic_sizes)))
ax.set_yticklabels([f"Topic {i}: {rep[:50]}..." 
                    for i, rep in zip(topic_sizes['Topic'], topic_sizes['Representation'])])
ax.set_xlabel('Number of Comments', fontsize=12)
ax.set_title(f'Topic Distribution - Polish (n={len(df_pl):,})', fontsize=14, fontweight='bold')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
save_figure(fig, 'pl_topics_overview.png', 'pl', 'visualizations')

# Save checkpoint
save_checkpoint(df_pl, 'pl', '05_topics_data')
update_pipeline_status('pl', 5, 'completed')

print('\n' + '='*70)
print('✓ BLOCK 5 COMPLETE - TOPIC MODELING DONE')
print('='*70)



Generating topic visualization...
✓ Saved: outputs\pl\visualizations\pl_topics_overview.png
✓ Checkpoint saved: 05_topics_data_pl.pkl
✓ Pipeline status updated: pl - Block 5 - completed

✓ BLOCK 5 COMPLETE - TOPIC MODELING DONE
